# Phase 4: Data Preprocessing and ML Dataset Preparation

**Project Title:** AI-Driven Intelligent Greywater Management and Smart Reuse Routing System  
**Objective:** Prepare the validated four-class routing dataset for supervised machine learning (XGBoost / Random Forest) while strictly enforcing data hygiene, preventing target leakage, and preserving physical interpretability.

> **CRITICAL DIRECTIVE:** Zero machine learning models are trained in this phase. The sole deliverable is clean, leakage-free, reproducible train/validation/test partitions ready for model training in Phase 5.


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 10})

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from models.preprocessing.preprocessor import GreywaterPreprocessor

print("Environment configured and Preprocessing modules loaded successfully.")


## 1. Load Routing Dataset
Load the derived four-class routing dataset generated and validated in Phase 3 (`dataset/processed/greywater_routing_labels.csv`).


In [ ]:
DATASET_PATH = '../dataset/processed/greywater_routing_labels.csv'
df = pd.read_csv(DATASET_PATH)

print(f"Loaded dataset from: {DATASET_PATH}")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
df.head()


## 2. Verify Routing Labels
Confirm the integrity of the target variable `Routing_Class` and `Routing_Class_Name` across all 1,500 samples.


In [ ]:
class_counts = df['Routing_Class'].value_counts().sort_index()
class_names = {0: "Sewer Bypass", 1: "Bio-filtration", 2: "Restricted Irrigation", 3: "Indoor Reuse"}
dist_summary = pd.DataFrame({
    'Routing_Class': class_counts.index,
    'Class_Name': [class_names[c] for c in class_counts.index],
    'Count': class_counts.values,
    'Percentage (%)': (class_counts.values / len(df) * 100).round(2)
})

print("Target Class Verification Summary:")
print(dist_summary)

assert df['Routing_Class'].isnull().sum() == 0, "Missing labels detected!"
assert set(df['Routing_Class'].unique()) == {0, 1, 2, 3}, "Unexpected class values!"
print("Routing label integrity 100% verified.")


## 3. Feature Selection
Define the candidate input features from domain-valid predictors:
- 1 categorical context variable: `Greywater_Source`
- 15 continuous physicochemical and biological sensor metrics:
  `pH, TEMP_C, SAL_ppt, TUR_NTU, DS_mg_L, TDS_mg_L, TSS_mg_L, COND_uS_cm, DO_mg_L, BOD_mg_L, COD_mg_L, NH4F_mg_L, NO3_mg_L, K_mg_L, E_coli_CFU_100mL`


In [ ]:
cat_cols = ['Greywater_Source']
num_cols = [
    'pH', 'TEMP_C', 'SAL_ppt', 'TUR_NTU', 'DS_mg_L', 'TDS_mg_L',
    'TSS_mg_L', 'COND_uS_cm', 'DO_mg_L', 'BOD_mg_L', 'COD_mg_L',
    'NH4F_mg_L', 'NO3_mg_L', 'K_mg_L', 'E_coli_CFU_100mL'
]
feature_candidates = cat_cols + num_cols
print(f"Candidate Features ({len(feature_candidates)}): {feature_candidates}")


## 4. Feature Leakage Audit
Strictly isolate and eliminate identifiers, duplicate labels, and post-decision metadata from feature matrix $X$:
- `Sample_ID` (Identifier - causes spurious memorization)
- `Routing_Class` (Target variable $y$)
- `Routing_Class_Name`, `Water_Quality_Class` (Direct target leakage)
- `Primary_Routing_Reason`, `Triggered_Parameters`, `Treatment_Required`, `Safety_Flag` (Direct decision leakage)


In [ ]:
leakage_path = '../reports/feature_leakage_audit.csv'
leakage_audit_df = pd.read_csv(leakage_path)
print(f"Loaded {len(leakage_audit_df)} feature audit evaluations:")
leakage_audit_df


## 5. Categorical Feature Handling
The single categorical feature `Greywater_Source` (Bathroom, Kitchen, Laundry, Mixed) is encoded via One-Hot Encoding into 4 binary indicators.
*Note: Label encoding is avoided to prevent imposing artificial ordinal rankings on nominal effluent sources.*


In [ ]:
print("Unique Categories in Greywater_Source:", df['Greywater_Source'].unique().tolist())


## 6. Numerical Feature Handling & Scaling Policy
**Scaling Decision for Tree-Based Models:**
- The primary target models for this project are **XGBoost** and **Random Forest**.
- Tree-based ensemble architectures are invariant to monotonic feature scaling; they partition feature space via orthogonal axis-aligned threshold splits.
- Preserving original physical units (mg/L, NTU, pH, CFU/100mL) maintains interpretability for domain engineers and SHAP attribution analyses.
- **Verdict:** Numerical predictors are passed through without arbitrary normalization or standard scaling.


In [ ]:
df[num_cols].describe().round(2)


## 7. Missing-Value Handling Strategy
- Audited missing values across all columns in `dataset/processed/greywater_routing_labels.csv`: **0 missing entries (0.00%)**.
- **Policy:** `"No imputation required for the current dataset."`
- The production preprocessing pipeline is designed defensively to handle missing data at inference time if needed.


In [ ]:
missing_counts = df[feature_candidates].isnull().sum()
print("Missing counts across candidate predictors:")
print(missing_counts[missing_counts > 0] if (missing_counts > 0).any() else "0 missing values across all features.")


## 8. Outlier Policy
- **Policy:** `"Outliers are retained unless a value is demonstrated to be erroneous."`
- Real domestic greywater exhibits severe contamination spikes (e.g., concentrated food waste washing in Kitchens or chemical detergent discharge in Laundry).
- Deleting or winsorizing these authentic extreme values would blind the machine learning models to the exact conditions that mandate `Sewer Bypass`.


## 9. Class Distribution & Imbalance
Analyze the operational class distribution to understand class imbalance prior to splitting.


In [ ]:
imbalance_path = '../reports/ml_class_distribution.csv'
imbalance_df = pd.read_csv(imbalance_path)
print("Class Distribution and Imbalance Ratios:")
imbalance_df


## 10. Stratified Train / Validation / Test Split (70 / 15 / 15)
To ensure rigorous validation and test benchmarking:
- **70% Training Set:** 1,050 samples
- **15% Validation Set:** 225 samples
- **15% Test Set:** 225 samples
- **Stratification:** Stratified by `Routing_Class` with fixed seed `random_state=42`.


In [ ]:
RANDOM_STATE = 42

def deterministic_stratified_split(data, target_col='Routing_Class', seed=42):
    rng = np.random.RandomState(seed)
    train_idx, val_idx, test_idx = [], [], []
    
    allocations = {
        0: (312, 67, 67),
        1: (505, 108, 109),
        2: (229, 49, 48),
        3: (4, 1, 1)
    }
    
    for c, (n_tr, n_va, n_te) in allocations.items():
        c_indices = data[data[target_col] == c].index.values.copy()
        rng.shuffle(c_indices)
        train_idx.extend(c_indices[:n_tr])
        val_idx.extend(c_indices[n_tr:n_tr+n_va])
        test_idx.extend(c_indices[n_tr+n_va:n_tr+n_va+n_te])
        
    train_out = data.loc[train_idx].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_out = data.loc[val_idx].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test_out = data.loc[test_idx].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return train_out, val_out, test_out

train_raw, val_raw, test_raw = deterministic_stratified_split(df, seed=RANDOM_STATE)
print(f"Partition Sizes: Train={len(train_raw)}, Validation={len(val_raw)}, Test={len(test_raw)}")


## 11. Preprocessing Pipeline Fitting
To prevent data leakage, the `GreywaterPreprocessor` is fitted **STRICTLY on the Training Set (`train_raw`)** and subsequently applied to transform validation and test sets.


In [ ]:
preprocessor = GreywaterPreprocessor()
preprocessor.fit(train_raw)

transformed_feature_names = preprocessor.get_feature_names_out()
print(f"Fitted Preprocessor on Training Set. Total Model Features ({len(transformed_feature_names)}):")
for i, fn in enumerate(transformed_feature_names, 1):
    print(f"  {i}. {fn}")


## 12. Transformation Validation & Feature Alignment
Transform the partitioned datasets and attach `Routing_Class` target labels.


In [ ]:
X_train = preprocessor.transform(train_raw)
X_val = preprocessor.transform(val_raw)
X_test = preprocessor.transform(test_raw)

train_processed = X_train.copy()
train_processed['Routing_Class'] = train_raw['Routing_Class'].values

val_processed = X_val.copy()
val_processed['Routing_Class'] = val_raw['Routing_Class'].values

test_processed = X_test.copy()
test_processed['Routing_Class'] = test_raw['Routing_Class'].values

print("Transformed Shapes:")
print(f"Train: {train_processed.shape} (19 features + 1 target)")
print(f"Val:   {val_processed.shape} (19 features + 1 target)")
print(f"Test:  {test_processed.shape} (19 features + 1 target)")


## 13. Save Processed ML Datasets
Export model-ready CSVs to `dataset/processed/` and archive serialized pipeline to `models/preprocessing/`.


In [ ]:
train_processed.to_csv('../dataset/processed/train.csv', index=False)
val_processed.to_csv('../dataset/processed/validation.csv', index=False)
test_processed.to_csv('../dataset/processed/test.csv', index=False)

# Save metadata audit splits (for provenance tracking)
train_raw.to_csv('../dataset/processed/metadata_train.csv', index=False)
val_raw.to_csv('../dataset/processed/metadata_val.csv', index=False)
test_raw.to_csv('../dataset/processed/metadata_test.csv', index=False)

# Save serialized pipeline
preprocessor.save('../models/preprocessing/preprocessor.pkl')
print("Successfully saved train.csv, validation.csv, test.csv, metadata splits, and preprocessor.pkl!")


## 14. Final Preprocessing Summary

1. **Leakage Prevention**: All non-predictive identifiers (`Sample_ID`), duplicate labels (`Routing_Class_Name`), and post-decision metadata were strictly excluded from model feature sets.
2. **Feature Stratification**: 19 model features produced (4 one-hot encoded source indicators + 15 physical parameters in raw units).
3. **Partition Stratification**: Exact 70% / 15% / 15% split preserving all 4 classes across all sets without data contamination (0 overlap).
4. **Readiness for Phase 5**: The machine learning datasets are validated, reproducible, and ready for model training.


In [ ]:
summary_path = '../reports/dataset_split_summary.csv'
summary_df = pd.read_csv(summary_path)
print("Partition Summary Table:")
summary_df
